# RolloTree: Tree Visualization & Inspection

This notebook covers:
1. Text export (`export_text`) — ASCII tree representation
2. Graphviz export (`export_graphviz`) — DOT format for rendering
3. Tree inspection (`apply`, `decision_path`, `get_n_leaves`, `get_depth`)

In [1]:
import pandas as pd
import numpy as np
from rollotree import RollingOCT, export_text, export_graphviz

In [2]:
# Load and prepare data
train = pd.read_csv("../rollotree/data/train.csv")
test = pd.read_csv("../rollotree/data/test.csv")

X_train = train.drop("y", axis=1)
y_train = train["y"]
X_test = test.drop("y", axis=1)
y_test = test["y"]

# Train models at different depths
model_d2 = RollingOCT(depth=2, solver="highs")
model_d2.fit(X_train, y_train)

model_d3 = RollingOCT(depth=3, solver="highs")
model_d3.fit(X_train, y_train)

print(f"Depth-2 test accuracy: {model_d2.score(X_test, y_test):.3f}")
print(f"Depth-3 test accuracy: {model_d3.score(X_test, y_test):.3f}")

Depth-2 test accuracy: 0.611
Depth-3 test accuracy: 0.778


## 1. Text Export

`export_text()` produces a human-readable tree representation, similar to sklearn's `export_text()`.

In [3]:
# Default: uses feature indices
print("=== Depth-2 Tree ===")
print(export_text(model_d2.tree_))

=== Depth-2 Tree ===
|--- feature_112 == 1
|   |--- feature_50 == 1
|   |   class: 2 {1: 0, 2: 2, 3: 0}
|   |--- feature_50 == 0
|   |   class: 3 {1: 0, 2: 0, 3: 12}
|--- feature_112 == 0
|   |--- feature_111 == 1
|   |   class: 3 {1: 0, 2: 0, 3: 18}
|   |--- feature_111 == 0
|   |   class: 2 {1: 53, 2: 62, 3: 13}


In [4]:
# With feature names from the DataFrame
feature_names = list(X_train.columns)
print("=== Depth-2 Tree (with feature names) ===")
print(export_text(model_d2.tree_, feature_names=feature_names))

=== Depth-2 Tree (with feature names) ===
|--- 112 == 1
|   |--- 50 == 1
|   |   class: 2 {1: 0, 2: 2, 3: 0}
|   |--- 50 == 0
|   |   class: 3 {1: 0, 2: 0, 3: 12}
|--- 112 == 0
|   |--- 111 == 1
|   |   class: 3 {1: 0, 2: 0, 3: 18}
|   |--- 111 == 0
|   |   class: 2 {1: 53, 2: 62, 3: 13}


In [5]:
# Depth-3 tree shows more structure
print("=== Depth-3 Tree ===")
print(export_text(model_d3.tree_, feature_names=feature_names))

=== Depth-3 Tree ===
|--- 112 == 1
|   |--- 50 == 1
|   |   class: 2 (pruned)
|   |--- 50 == 0
|   |   class: 3 (pruned)
|--- 112 == 0
|   |--- 31 == 1
|   |   |--- 21 == 1
|   |   |   class: 2 (pruned)
|   |   |--- 21 == 0
|   |   |   class: 1 (pruned)
|   |--- 31 == 0
|   |   |--- 111 == 1
|   |   |   class: 3 {1: 0, 2: 0, 3: 18}
|   |   |--- 111 == 0
|   |   |   class: 2 {1: 36, 2: 56, 3: 13}


## 2. Graphviz Export

`export_graphviz()` produces a DOT string that can be rendered with Graphviz.

In [6]:
dot = export_graphviz(model_d2.tree_, feature_names=feature_names)
print(dot)

digraph Tree {
    node [shape=box, style="filled, rounded"];
    1 [label="112 == ?", fillcolor="#f9e79f"];
    2 [label="50 == ?", fillcolor="#f9e79f"];
    4 [label="class: 2\nsamples: 2\nvalue: [0, 2, 0]", fillcolor="#aed6f1"];
    2 -> 4 [label="= 1"];
    5 [label="class: 3\nsamples: 12\nvalue: [0, 0, 12]", fillcolor="#aed6f1"];
    2 -> 5 [label="= 0"];
    1 -> 2 [label="= 1"];
    3 [label="111 == ?", fillcolor="#f9e79f"];
    6 [label="class: 3\nsamples: 18\nvalue: [0, 0, 18]", fillcolor="#aed6f1"];
    3 -> 6 [label="= 1"];
    7 [label="class: 2\nsamples: 128\nvalue: [53, 62, 13]", fillcolor="#aed6f1"];
    3 -> 7 [label="= 0"];
    1 -> 3 [label="= 0"];
}


In [7]:
# With class names for readability
dot = export_graphviz(
    model_d2.tree_,
    feature_names=feature_names,
    class_names=["Barolo", "Grignolino", "Barbera"],
)
print(dot)

digraph Tree {
    node [shape=box, style="filled, rounded"];
    1 [label="112 == ?", fillcolor="#f9e79f"];
    2 [label="50 == ?", fillcolor="#f9e79f"];
    4 [label="class: 2\nsamples: 2\nvalue: [0, 2, 0]", fillcolor="#aed6f1"];
    2 -> 4 [label="= 1"];
    5 [label="class: 3\nsamples: 12\nvalue: [0, 0, 12]", fillcolor="#aed6f1"];
    2 -> 5 [label="= 0"];
    1 -> 2 [label="= 1"];
    3 [label="111 == ?", fillcolor="#f9e79f"];
    6 [label="class: 3\nsamples: 18\nvalue: [0, 0, 18]", fillcolor="#aed6f1"];
    3 -> 6 [label="= 1"];
    7 [label="class: 2\nsamples: 128\nvalue: [53, 62, 13]", fillcolor="#aed6f1"];
    3 -> 7 [label="= 0"];
    1 -> 3 [label="= 0"];
}


In [8]:
# Render inline if graphviz is installed
try:
    import graphviz
    dot_src = export_graphviz(
        model_d3.tree_,
        feature_names=feature_names,
        class_names=["Barolo", "Grignolino", "Barbera"],
    )
    display(graphviz.Source(dot_src))
except ImportError:
    print("Install graphviz to render: pip install graphviz")
    print("Then: brew install graphviz  (or apt install graphviz)")

Install graphviz to render: pip install graphviz
Then: brew install graphviz  (or apt install graphviz)


## 3. Tree Inspection

### 3.1 `apply()` — Leaf Node Assignment

Returns the leaf node ID that each sample is routed to.

In [9]:
leaf_ids = model_d3.apply(X_test)

print(f"Shape: {leaf_ids.shape}")
print(f"Unique leaf IDs: {np.unique(leaf_ids)}")
print(f"\nFirst 10 samples -> leaves: {leaf_ids[:10]}")

# Distribution of samples across leaves
unique, counts = np.unique(leaf_ids, return_counts=True)
print("\nLeaf distribution:")
for lid, cnt in zip(unique, counts):
    print(f"  Leaf {lid}: {cnt} samples")

Shape: (18,)
Unique leaf IDs: [ 5 12 13 14 15]

First 10 samples -> leaves: [13 13 15 15 15 13 12 15 15 15]

Leaf distribution:
  Leaf 5: 3 samples
  Leaf 12: 1 samples
  Leaf 13: 3 samples
  Leaf 14: 1 samples
  Leaf 15: 10 samples


### 3.2 `decision_path()` — Node Traversal

Returns a sparse matrix showing which nodes each sample passes through.

In [10]:
path = model_d3.decision_path(X_test)

print(f"Type: {type(path)}")
print(f"Shape: {path.shape}  (n_samples x n_nodes)")
print(f"Non-zeros: {path.nnz}")

# Show path for first sample
sample_path = path[0].toarray().flatten()
visited = np.where(sample_path == 1)[0]
print(f"\nSample 0 visits nodes: {visited}")

Type: <class 'scipy.sparse._csr.csr_matrix'>
Shape: (18, 16)  (n_samples x n_nodes)
Non-zeros: 69

Sample 0 visits nodes: [ 1  3  6 13]


### 3.3 `get_n_leaves()` and `get_depth()`

In [11]:
print(f"Depth-2 model:")
print(f"  Leaves: {model_d2.get_n_leaves()}")
print(f"  Depth:  {model_d2.get_depth()}")

print(f"\nDepth-3 model:")
print(f"  Leaves: {model_d3.get_n_leaves()}")
print(f"  Depth:  {model_d3.get_depth()}")

Depth-2 model:
  Leaves: 4
  Depth:  2

Depth-3 model:
  Leaves: 2
  Depth:  3


### 3.4 Manual Tree Traversal

You can also inspect `model.tree_` directly.

In [12]:
tree = model_d3.tree_

print(f"Branch nodes: {sorted(tree.branch_nodes.keys())}")
print(f"Leaf nodes:   {sorted(tree.leaf_nodes.keys())}")
print(f"Pruned nodes: {sorted(tree._pruned_node_ids)}")

print("\nBranch splits:")
for nid, node in sorted(tree.branch_nodes.items()):
    if node.feature_index is not None:
        fname = feature_names[node.feature_index] if node.feature_index < len(feature_names) else f"f{node.feature_index}"
        print(f"  Node {nid}: {fname}")

print("\nLeaf predictions:")
for lid, leaf in sorted(tree.leaf_nodes.items()):
    if leaf.predicted_class is not None:
        pruned = " (pruned)" if leaf.is_pruned else ""
        dist = dict(leaf.class_distribution) if leaf.class_distribution else {}
        print(f"  Leaf {lid}: class={leaf.predicted_class} dist={dist}{pruned}")

Branch nodes: [1, 2, 3, 6, 7]
Leaf nodes:   [4, 5, 12, 13, 14, 15]
Pruned nodes: [4, 5, 12, 13]

Branch splits:
  Node 1: 113
  Node 2: 51
  Node 3: 32
  Node 6: 22
  Node 7: 112

Leaf predictions:
  Leaf 4: class=2 dist={} (pruned)
  Leaf 5: class=3 dist={} (pruned)
  Leaf 12: class=2 dist={} (pruned)
  Leaf 13: class=1 dist={} (pruned)
  Leaf 14: class=3 dist={1: 0, 2: 0, 3: 18}
  Leaf 15: class=2 dist={1: 36, 2: 56, 3: 13}


## Summary

| Tool | Code |
|------|------|
| ASCII tree | `export_text(model.tree_, feature_names=names)` |
| Graphviz DOT | `export_graphviz(model.tree_, feature_names=names)` |
| Leaf assignments | `model.apply(X)` |
| Decision path | `model.decision_path(X)` → sparse CSR |
| Number of leaves | `model.get_n_leaves()` |
| Actual depth | `model.get_depth()` |

See **01_quickstart.ipynb** for basics, **03_sklearn_integration.ipynb** for sklearn interop.